# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'nebula platform',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'social media: LinkedIn',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media: Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media: Facebook',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'endpoints product page',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'chat product page', 'url': 'https://huggingface.co/chat'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
moonshotai/Kimi-K2.5
Updated
3 days ago
•
96.2k
•
1.45k
Tongyi-MAI/Z-Image
Updated
5 days ago
•
6.35k
•
794
tencent/HunyuanImage-3.0-Instruct
Updated
5 days ago
•
148
•
773
deepseek-ai/DeepSeek-OCR-2
Updated
4 days ago
•
144k
•
628
nvidia/personaplex-7b-v1
Updated
5 days ago
•
101k
•
1.58k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.14k
Qwen3-TTS Demo
🎙
1.14k
Generate realistic speech from text with custom voices or voice cloning
Running
on
Zero
MCP
1.95k
Z Image Turbo
🖼
1.95k
Generate stunning images from text descriptions i

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2.5\nUpdated\n3 days ago\n•\n96.2k\n•\n1.45k\nTongyi-MAI/Z-Image\nUpdated\n5 days ago\n•\n6.35k\n•\n794\ntencent/HunyuanImage-3.0-Instruct\nUpdated\n5 days ago\n•\n148\n•\n773\ndeepseek-ai/DeepSeek-OCR-2\nUpdated\n4 days ago\n•\n144k\n•\n628\nnvidia/personaplex-7b-v1\nUpdated\n5 days ago\n•\n101k\n•\n1.58k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.14k\nQwen

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

## About Hugging Face
Hugging Face is the premier collaboration platform for the machine learning (ML) community, dedicated to building the future of Artificial Intelligence (AI). As a central hub for open-source ML, Hugging Face empowers engineers, scientists, and end users worldwide to share, discover, and experiment with an extensive repository of models, datasets, and AI applications. The company is committed to fostering an open and ethical AI future by nurturing an active, fast-growing community.

## What We Offer
- **2M+ Machine Learning Models:** Access and contribute to an extensive library covering diverse AI modalities including text, image, video, audio, and even 3D.
- **500k+ Datasets:** Utilize a comprehensive collection of datasets to train, test, and improve machine learning models.
- **Spaces:** Host and explore AI applications on an easy-to-use platform, enabling interactive demos and prototypes running on the Hugging Face infrastructure.
- **Community Hub:** Collaborate and learn alongside thousands of AI researchers, developers, and enthusiasts to accelerate innovation.
- **Open Source Stack:** Leverage the free, open-source tools and libraries that enable rapid ML development and deployment.
- **Enterprise Solutions:** For organizations requiring tailored support and computation, Hugging Face offers paid compute services and enterprise-grade solutions.

## Why Choose Hugging Face?
- **Collaborative Innovation:** Build and share in a vibrant community that encourages transparency, peer support, and shared progress.
- **Portfolio Building:** Showcase your work to the world and establish your profile in the global ML landscape.
- **Multi-Modal AI:** Explore and create models across various data modalities, expanding the possibilities of AI applications.
- **Ethical AI:** Commitment to developing AI responsibly and inclusively, fostering openness and fairness.
- **Continuous Learning:** Access extensive documentation and community resources to grow your AI expertise.

## Customers and Users
Hugging Face serves a diverse array of users, from independent researchers and AI hobbyists to academic institutions and leading tech companies. Popular trending models and datasets on the platform demonstrate active contributions from renowned organizations such as NVIDIA, Tencent, and MoonshotAI. The platform supports innovation in fields like speech synthesis, computer vision, natural language processing, and beyond.

## Company Culture
- **Community-Centric:** Puts the AI community first by prioritizing shared knowledge and collaboration.
- **Open Source Advocates:** Strong believers in transparency and the power of open science.
- **Innovation Driven:** Encourages creativity, learning, and pushing boundaries through experimentation.
- **Ethics and Responsibility:** Dedicated to ethical AI development with global impact.

## Careers at Hugging Face
Join a passionate team at the forefront of AI innovation:

- **Roles:** Openings across engineering, research, product management, and community engagement.
- **Environment:** Dynamic, inclusive workplace focused on impact, learning, and collaboration.
- **Opportunity:** Work with cutting-edge ML tech, contribute to open-source projects, and influence the AI ecosystem.
- **Apply:** Visit Hugging Face’s career page to explore current job openings and opportunities to build the future of AI.

---

**Connect with Hugging Face**  
- Website: https://huggingface.co  
- Join the community to innovate, learn, and shape the next generation of machine learning.

---

*Hugging Face – The AI community building the future.*

In [19]:
create_brochure("Anandians Connections", "https://anandians.com/")

Selecting relevant links for https://anandians.com/ by calling gpt-5-nano
Found 11 relevant links


# Anandians Connections - Company Brochure

---

## About Anandians Connections

Anandians Connections (O.P.C) Private Limited is a cutting-edge AI Semiconductor startup headquartered in RajPur, Rohtas, Bihar, India. Rooted deeply in India’s rich heritage, the company draws inspiration from ancient Aryan and Dravidian civilizations to bridge the gap between tradition and futuristic technology. With a strong foundation in research and innovation, Anandians Connections is committed to delivering world-class AI and semiconductor solutions that empower businesses and individuals globally.

---

## Our Mission

- To inspire, innovate, and involve people worldwide.
- To help users optimize their time, making it both enjoyable and productive.
- To transform ideas from initial concepts to globally recognized, high-impact solutions.
- To ethically lead in AI and semiconductor development by blending time-honored wisdom with state-of-the-art technological breakthroughs.

---

## Our Core Offerings

Anandians operates at the intersection of established enterprises and emerging innovators, serving a diverse client base, including corporates, global agencies, startups, and creative studios.

Key product and service highlights:

- **UFREE Gen 1.0**  
- **SNMGS GEN 3.O**  
- **DA.OPS VERSION 2.O** – Traditional Technical Services delivering hands-on IT and technical support.  
- **NAPP.LCP INDUSTRY 4.0** – New Age Project Production Lifecycle Pipeline, a modern and innovative methodology for project conception, development, and execution.

Additional bespoke services to tailor solutions for industry-specific challenges.

---

## Company Culture

Anandians fosters a culture steeped in research, innovation, and entrepreneurial spirit. The company values:

- **Innovation-driven mindset:** Continuously pushing boundaries in AI and semiconductor technology.
- **Heritage-inspired vision:** Drawing from India’s ancient civilizations to guide futuristic technology development.
- **Collaboration:** Working alongside global partners and startups to co-create impactful outcomes.
- **Ethical Responsibility:** Commitment to creating world-class technology solutions with ethical considerations.

The team thrives on transforming complex ideas into real-world applications that resonate globally.

---

## Clientele

Anandians engages a wide range of customers:

- Established corporations seeking advanced AI and semiconductor integration.
- Global agencies requiring innovative technology solutions.
- Startups and creative studios looking for cutting-edge product development support.

By harmonizing traditional knowledge with disruptive technology, Anandians supports clients in achieving technological excellence and market leadership.

---

## Careers at Anandians Connections

Anandians Connections invites passionate, innovative, and research-oriented professionals to join their journey. With a 24/7 operational ethos and a focus on sustainability and growth, career opportunities include roles in:

- AI Research and Development
- Semiconductor Engineering
- Software and Platform Development
- Project Management (particularly in Industry 4.0 methodologies)
- Technical Support and Operations

Candidates eager to contribute to a future-focused company grounded in heritage and innovation can reach out via connect@anandians.com or call +91 9870603314.

---

## Contact Us

**Anandians Connections (O.P.C) Private Limited**  
Address: RajPur, Rohtas, Bihar, India, Asia  
Phone: +91 9870603314  
Email: connect@anandians.com  
Availability: 24/7  

Follow us on our digital platforms to stay updated on the latest innovations and opportunities.

---

Anandians Connections — From the edge of the universe to the palm of your hand, bridging heritage with the future of technology.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community and collaboration platform dedicated to building the future of machine learning (ML). It is the go-to place where ML enthusiasts, researchers, and developers from around the world come together to share, discover, and collaborate on models, datasets, applications, and cutting-edge AI technologies.

---

## What We Offer

- **Massive Model Repository:** Explore and browse over **2 million open-source ML models** spanning modalities like text, image, audio, video, and even 3D.
- **Extensive Datasets:** Access a collection of **500,000+ datasets** to train and validate your AI models.
- **Spaces:** An innovative platform to run and showcase AI applications live, offering demos such as text-to-speech, image generation, 3D camera controls, and more.
- **Community Hub:** Join a fast-growing vibrant community collaborating openly to advance AI and machine learning.
- **Open Source Stack:** Move faster in your AI projects using Hugging Face’s open-source libraries and tools.
- **Compute Resources:** Accelerate your workflows with paid compute and enterprise-grade solutions.

---

## Our Vision and Culture

Hugging Face is centered on **open, ethical AI development**. The platform empowers the next generation of machine learning engineers, scientists, and end users to build, learn, and share their work in a transparent and collaborative environment. 

The company culture thrives on:
- **Community-driven innovation**
- **Transparency and openness**
- **Ethical AI principles**
- **Accessibility and inclusivity in AI education and development**

---

## Our Customers and Community

Hugging Face serves a diverse audience including:
- Individual AI and ML practitioners building portfolios and experimenting with models.
- Research institutes and academic communities advancing AI science.
- Industry leaders and enterprises adopting AI technologies at scale.
- Developers creating innovative AI applications across multiple domains.

Notably, models and datasets created or shared on Hugging Face come from top contributors, including renowned organizations like NVIDIA and Tencent.

---

## Careers at Hugging Face

Join a mission-driven company revolutionizing the AI landscape! Careers at Hugging Face offer opportunities to:
- Work alongside world-class AI researchers and engineers.
- Contribute to open-source projects impacting millions globally.
- Innovate in a fast-growing startup environment with a strong focus on ethical AI.
- Help build tools and infrastructure that democratize AI access.

To explore current openings or internships, visit the Hugging Face careers page on their website.

---

## Connect with Hugging Face

- **Website:** [huggingface.co](https://huggingface.co)
- **Explore Models, Datasets & Spaces**
- **Join the Community and Collaborate**

---

### Branding

- Official colors: yellow (#FFD21E), orange (#FF9D00), grey (#6B7280)
- Open-source logo assets available in SVG, PNG, AI formats

---

Embrace the future of AI with Hugging Face — **The AI community building the future.**

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community and collaboration platform building the future of machine learning. Serving as the central hub for machine learning engineers, scientists, and AI enthusiasts worldwide, Hugging Face empowers users to share, explore, discover, and experiment with over **2 million models**, **500,000+ datasets**, and **1 million+ applications** across modalities such as text, image, video, audio, and even 3D.

Their open source ecosystem and fast-growing community place Hugging Face at the heart of the AI revolution, fostering innovation in an ethical, inclusive, and collaborative manner. The platform enables users to host unlimited public models, datasets, and applications, accelerating ML development and providing a space to build personal and organizational portfolios.

---

## What Hugging Face Offers

### Community & Collaboration
- A global machine learning community collaborating on advanced AI models and datasets.
- A platform for building and sharing innovative AI applications known as "Spaces."
- Tools to build a personal ML profile and portfolio visible to the world.
- Access to cutting-edge open-source ML libraries and tools.

### Extensive Resources
- **2M+ models:** Covering a diverse range of ML tasks with frequent updates and trending models.
- **500k+ datasets:** Curated for training and benchmarking across many use cases.
- Ready-to-use demos and playgrounds for experimenting with AI models.

### Enterprise Solutions
Hugging Face offers a robust Enterprise Hub designed for organizations that need to scale AI initiatives with:
- Enterprise-grade security features including Single Sign-On (SSO), granular access controls, audit logs, and token management.
- Private storage and dataset viewers for confidential data collaboration.
- Advanced compute resources with ZeroGPU quota boosts and customizable inference providers.
- Analytics dashboards for monitoring and optimizing model usage.
- Flexible subscription plans ($20/user/month starting for Team) and customizable contracts for Enterprise clients.

---

## Company Culture

- **Open and Ethical AI:** Hugging Face is dedicated to building AI technologies responsibly, encouraging transparency and community stewardship.
- **Collaboration:** The platform is designed to enable seamless collaboration across individuals and organizations worldwide.
- **Innovation-driven:** With a talented science team pushing technology boundaries and a thriving developer ecosystem, Hugging Face constantly explores the cutting edge of AI research.
- **Community-first:** The heart of the company lies in its engaged community of machine learning engineers, data scientists, researchers, and AI enthusiasts.

---

## Careers

Hugging Face’s rapid growth and prominent position in AI make it an exciting place for talent passionate about machine learning and AI ethics to join. They seek professionals eager to work in a collaborative environment at the forefront of the AI revolution. Career openings typically span roles in research, engineering, product management, and community engagement.

Interested candidates are encouraged to explore Hugging Face’s Careers page for current opportunities to innovate alongside top-tier AI scientists and engineers.

---

## Why Choose Hugging Face?

- Access the fastest-growing ML community and ecosystem worldwide.
- Collaborate on and deploy state-of-the-art machine learning models easily.
- Leverage enterprise-grade security and scalable infrastructure.
- Contribute to a mission-driven company building an open, ethical AI future.

---

## Get Started

Join millions of machine learning engineers and AI developers shaping the future of AI at Hugging Face.  
Sign up today at [huggingface.co](https://huggingface.co) to explore models, datasets, and applications or to launch your own AI projects.

---

## Contact and Social

- Website: https://huggingface.co  
- GitHub, Twitter, LinkedIn, Discord: Find Hugging Face’s active presence online  
- Enterprise Sales: Contact for tailored AI platform solutions  

---

**Hugging Face** – The AI community building the future.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>